# JuaKazi `ki-bias-classifier-v1` — Training Notebook

**Model:** `juakazike/ki-bias-classifier-v1`  
**Base:** `Davlan/afro-xlmr-large-76L` · **Task:** Kikuyu gender bias (binary)  
**Runtime:** GPU T4 · ~4 hours  
**Target:** BIAS Precision ≥ 0.65, Recall ≥ 0.65, F1 ≥ 0.70

### Why afro-xlmr-large-76L?

`Davlan/afro-xlmr-base` does **not** cover Kikuyu (Gikuyu/kik).  
`Davlan/afro-xlmr-large-76L` is the only AfroXLM-R variant that includes Kikuyu.  
This is the single most impactful change for KI — the base model alone should  
lift recall from 0.510 (rules-only) toward 0.70+.

### Data source (11,622 rows)

| File | Rows | Notes |
|---|---|---|
| `eval/ground_truth_ki_v8.csv` | 11,622 | 1,603 biased / 10,019 neutral, all qa_status=passed |

All rows have `expected_correction` — same file feeds the KI corrector notebook.

### Memory note
76L is ~560M params. On T4 (16GB VRAM):  
- batch_size=16 + gradient_accumulation=2 (effective 32) → fits  
- If OOM: reduce batch to 8, increase gradient_accumulation to 4

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `ground_truth_ki_v8.csv` to Drive at `MyDrive/juakazi/`
3. Add `HF_TOKEN` to Colab Secrets


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'transformers>=4.38.0', 'tokenizers>=0.15.0', 'accelerate>=0.27.0',
    'scikit-learn>=1.4.0', 'huggingface_hub>=0.20.0', 'numpy<2.0.0',
], capture_output=True, text=True)
print('Install OK.' if result.returncode == 0 else f'FAILED:\n{result.stderr}')
print('Restart runtime, then run from Cell 2.')

In [ ]:
# ── Cell 2: Verify environment ───────────────────────────────────────────────
import torch, transformers, numpy as np
print(f'torch: {torch.__version__}  |  transformers: {transformers.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'No GPU — go to Runtime → Change runtime type → T4 GPU'

In [ ]:
# ── Cell 3: Mount Drive + load KI ground truth ───────────────────────────────
# Kaggle: files at /kaggle/input/{dataset-slug}/

import os, csv
GT_CSV = '/kaggle/input/juakazi-ki-training/ground_truth_ki_v8.csv'
assert os.path.exists(GT_CSV), f'Not found: {GT_CSV}'

with open(GT_CSV, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

# Filter to rows with text and valid has_bias
all_data   = []
seen_texts = set()
for r in rows:
    text = r.get('text', '').strip()
    hb   = str(r.get('has_bias', '')).strip().lower()
    if not text or text in seen_texts: continue
    if hb not in ('true', 'false', '1', '0'): continue
    seen_texts.add(text)
    all_data.append((text, 1 if hb in ('true', '1') else 0))

bias_data    = [(t, l) for t, l in all_data if l == 1]
neutral_data = [(t, l) for t, l in all_data if l == 0]

print(f'Total: {len(all_data):,}  |  Biased: {len(bias_data):,}  |  Neutral: {len(neutral_data):,}')
print(f'Ratio: 1:{len(neutral_data)//max(len(bias_data),1)}')

In [ ]:
# ── Cell 4: Config ───────────────────────────────────────────────────────────
import random, numpy as np
from pathlib import Path

SEED          = 42
# KEY DIFFERENCE from HA/ZU: use 76L — the only base model covering Kikuyu
BASE_MODEL    = 'Davlan/afro-xlmr-large-76L'
OUTPUT_DIR    = '/kaggle/working/output'
MAX_LEN       = 128

n_bias, n_neutral = len(bias_data), len(neutral_data)
raw_ratio         = n_neutral / max(n_bias, 1)
POS_WEIGHT        = min(raw_ratio, 15.0)  # cap at 15x

TRAIN_SPLIT   = 0.80
VAL_SPLIT     = 0.10
EPOCHS        = 10
# Smaller batch + gradient accumulation for 76L (larger model)
BATCH         = 8
GRAD_ACCUM    = 4   # effective batch = 32
LR            = 1e-5  # lower LR for large model
WARMUP_RATIO  = 0.10
WEIGHT_DECAY  = 0.01
FREEZE_LAYERS = 8   # freeze more layers for 76L (24 total vs 12 in base)
REPO_ID       = 'juakazike/ki-bias-classifier-v1'

random.seed(SEED); np.random.seed(SEED)
import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Base model: {BASE_MODEL}  (76L — covers Kikuyu)')
print(f'n_bias={n_bias:,}  n_neutral={n_neutral:,}  POS_WEIGHT={POS_WEIGHT:.1f}')
print(f'Batch={BATCH}  GradAccum={GRAD_ACCUM}  EffectiveBatch={BATCH*GRAD_ACCUM}')
print(f'REPO={REPO_ID}')

In [ ]:
# ── Cell 5: Build splits ─────────────────────────────────────────────────────
import random

combined = all_data[:]
random.shuffle(combined)

n       = len(combined)
n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)

train_data = combined[:n_train]
val_data   = combined[n_train:n_train + n_val]
test_data  = combined[n_train + n_val:]

print(f'Train: {len(train_data):,}  ({sum(1 for _,l in train_data if l==1):,} bias)')
print(f'Val:   {len(val_data):,}  ({sum(1 for _,l in val_data   if l==1):,} bias)')
print(f'Test:  {len(test_data):,}  ({sum(1 for _,l in test_data  if l==1):,} bias)  ← held out')

In [ ]:
# ── Cell 6: Tokenizer + Dataset ──────────────────────────────────────────────
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

print(f'Loading tokenizer from {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class BiasDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data; self.tokenizer = tokenizer; self.max_len = max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer(text, truncation=True, max_length=self.max_len,
                             padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(label, dtype=torch.long)}

train_ds = BiasDataset(train_data, tokenizer, MAX_LEN)
val_ds   = BiasDataset(val_data,   tokenizer, MAX_LEN)
test_ds  = BiasDataset(test_data,  tokenizer, MAX_LEN)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
# ── Cell 7: Model (76L) + WeightedTrainer ────────────────────────────────────
import torch, numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

print(f'Loading {BASE_MODEL} (~560M params, may take 2-3 min)...')
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0: 'NEUTRAL', 1: 'BIAS'}, label2id={'NEUTRAL': 0, 'BIAS': 1},
    ignore_mismatched_sizes=True,
)

# Freeze bottom FREEZE_LAYERS layers (76L has 24 encoder layers)
frozen = 0
for i, layer in enumerate(model.roberta.encoder.layer):
    if i < FREEZE_LAYERS:
        for p in layer.parameters(): p.requires_grad = False
        frozen += 1

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Frozen: {frozen}/{len(model.roberta.encoder.layer)} layers  |  Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weight = torch.tensor([1.0, POS_WEIGHT], dtype=torch.float, device=outputs.logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weight)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = int(((preds==1)&(labels==1)).sum()); fp = int(((preds==1)&(labels==0)).sum())
    fn = int(((preds==0)&(labels==1)).sum())
    p = tp/(tp+fp) if (tp+fp)>0 else 0; r = tp/(tp+fn) if (tp+fn)>0 else 0
    f = 2*p*r/(p+r) if (p+r)>0 else 0
    print(f'  TP={tp} FP={fp} FN={fn} | P={p:.3f} R={r:.3f} F1={f:.3f}')
    return {'f1': f, 'precision': p, 'recall': r}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1', greater_is_better=True, logging_steps=50,
    fp16=torch.cuda.is_available(), seed=SEED, report_to='none',
)
trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print('Trainer ready.')

In [ ]:
# ── Cell 8: TRAIN ───────────────────────────────────────────────────────────
# Expected: ~4h on T4. Save checkpoints to Drive to survive disconnects.
result = trainer.train()
print(f'Done. Steps={result.global_step}  Loss={result.training_loss:.4f}')

In [ ]:
# ── Cell 9: Evaluate on TEST SET ────────────────────────────────────────────
import numpy as np, torch, torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score

pred_out = trainer.predict(test_ds)
preds    = np.argmax(pred_out.predictions, axis=-1)
labels   = pred_out.label_ids

print('=== TEST SET RESULTS ===')
print(classification_report(labels, preds, target_names=['NEUTRAL', 'BIAS']))

probs   = F.softmax(torch.tensor(pred_out.predictions), dim=-1)[:, 1].numpy()
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.20, 0.90, 0.01):
    f = f1_score(labels, (probs >= t).astype(int), average='binary')
    if f > best_f1: best_f1, best_t = f, float(t)

target_met = best_f1 >= 0.70
print(f'\nBest threshold: {best_t:.2f}  Test F1: {best_f1:.4f}  (target ≥0.70)  {"✓ TARGET MET" if target_met else "✗ BELOW TARGET"}')
print(f'>>> Set: JUAKAZI_KI_THRESHOLD={best_t:.2f}')

In [ ]:
# ── Cell 10: Save + upload to HuggingFace ───────────────────────────────────
import json
from sklearn.metrics import precision_score, recall_score
from huggingface_hub import HfApi
import os

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

opt_preds = (probs >= best_t).astype(int)
meta = {
    'model_id': REPO_ID, 'base_model': BASE_MODEL, 'language': 'ki',
    'test_f1': round(best_f1, 4),
    'test_precision': round(float(precision_score(labels, opt_preds, zero_division=0)), 4),
    'test_recall': round(float(recall_score(labels, opt_preds, zero_division=0)), 4),
    'threshold': best_t, 'target_met': target_met,
    'train_size': len(train_data), 'test_size': len(test_data),
    'n_bias': n_bias, 'n_neutral': n_neutral, 'pos_weight': POS_WEIGHT,
    'note': '76L base — only model covering Kikuyu (kik) language',
}
with open(f'{OUTPUT_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

try:
    HF_TOKEN = os.environ.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN and target_met:
    api = HfApi()
    api.create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True, private=False)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, token=HF_TOKEN)
    print(f'Uploaded → https://huggingface.co/{REPO_ID}')
    print(f'Set: JUAKAZI_KI_MODEL={REPO_ID}  JUAKAZI_KI_THRESHOLD={best_t:.2f}')
elif not target_met:
    print('⚠️  Not uploading — target not met. Try reducing FREEZE_LAYERS or lowering LR.')
else:
    print('Set HF_TOKEN in Colab Secrets to upload.')